# NB_LIBRO_11 · Figuras y experimentos del capítulo 9

**Libro:** *AgroVisión: del píxel a la decisión* · capítulo 9, «Del fotograma al comportamiento»

Este notebook:

1. lee el registro de seguimiento que el proyecto guardó (192 167 filas, una por ave y fotograma) y examina el video de 20 segundos fotograma a fotograma;
2. estudia la fragmentación de las identidades: dónde nacen y dónde terminan los identificadores, y cuántos son la misma ave vista dos veces;
3. dibuja las trayectorias y sigue a varias aves durante los 20 segundos;
4. mide el movimiento con distintos intervalos de tiempo, para separar el desplazamiento real del temblor de las cajas;
5. calcula la ocupación por zonas y el mapa de concentración;
6. hace un experimento nuevo: repite el seguimiento con ByteTrack, BoT-SORT y varias configuraciones de cada uno.

**Usa (solo lectura):** de `AgroUTMACH_PIX`, el video `04_VIDEOS_FERIA/PIO_tracking_estable_original_20s.mp4`, el registro `05_RESULTADOS/NB04_TRACKING/NB04_tracks_por_frame.csv` y el modelo `03_MODELOS/AgroUTMACH_PIX_YOLO11s_PIO_best.pt`. No modifica nada del proyecto.

**Produce:** seis figuras, `valores_cap09.json` y varias tablas, en `AGROVISION_LIBRO`. Al final descarga un zip.

**Cómo ejecutarlo:** *Entorno de ejecución → Cambiar tipo de entorno → GPU* (L4 recomendada) y *Ejecutar todo*. Tarda de 12 a 20 minutos; casi todo es el experimento del final. Si Colab se desconecta, vuelva a ejecutar todo: las configuraciones terminadas se reutilizan. Suba al chat el zip que se descarga al final.

In [ ]:
%pip install -q ultralytics==8.4.172

In [ ]:
import os, re, json, math, zipfile, time, shutil, gc, traceback, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

BASE = Path(os.environ.get('AGROVISION_BASE', '/content/drive/MyDrive'))
LOCAL = Path(os.environ.get('AGROVISION_LOCAL', '/content'))
PIX = BASE / 'AgroUTMACH_PIX'
VIDEO_DRIVE = PIX / '04_VIDEOS_FERIA' / 'PIO_tracking_estable_original_20s.mp4'
TRACKS_CSV = PIX / '05_RESULTADOS' / 'NB04_TRACKING' / 'NB04_tracks_por_frame.csv'
MOD_PIO = PIX / '03_MODELOS' / 'AgroUTMACH_PIX_YOLO11s_PIO_best.pt'
DIR_FIG = BASE / 'AGROVISION_LIBRO' / '02_FIGURAS' / 'cap09'
DIR_VAL = BASE / 'AGROVISION_LIBRO' / '03_VALORES'
DIR_EXP = DIR_VAL / 'cap09_seguimiento'
for d in (DIR_FIG, DIR_VAL, DIR_EXP):
    d.mkdir(parents=True, exist_ok=True)
for p in (VIDEO_DRIVE, TRACKS_CSV, MOD_PIO):
    print('OK   ' if p.exists() else 'FALTA', p)
assert VIDEO_DRIVE.exists() and TRACKS_CSV.exists(), 'Faltan el video o el registro de seguimiento del proyecto.'

REPETIR_SEGUIMIENTO = True   # ponga False para omitir el experimento del final (unos 10 minutos)

import torch, ultralytics
from ultralytics import YOLO
HAY_GPU = torch.cuda.is_available()
DISP = 0 if HAY_GPU else 'cpu'
GPU = torch.cuda.get_device_name(0) if HAY_GPU else 'sin GPU'
print('GPU:', GPU, '| PyTorch', torch.__version__, '| Ultralytics', ultralytics.__version__)

VIDEO = LOCAL / 'cap09_video_20s.mp4'
if not VIDEO.exists() or VIDEO.stat().st_size != VIDEO_DRIVE.stat().st_size:
    shutil.copy(VIDEO_DRIVE, VIDEO)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from PIL import Image, ImageOps
from skimage.color import rgb2lab, lab2rgb, rgb2hsv

ANCHO_CM = 16.0                      # ancho de caja de texto del libro
ANCHO_IN = ANCHO_CM / 2.54
DPI = 300

# Tinta y colores de apoyo. Los datos categóricos no se distinguen solo por color:
# las cinco clases se muestran en paneles separados (Figura 2.7).
TINTA = '#0b0b0b'
TINTA_2 = '#52514e'
GRIS_FONDO = '#c9c8c2'
ACENTO = '#2a78d6'
REJILLA = '#e4e3de'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.edgecolor': TINTA_2,
    'axes.labelcolor': TINTA,
    'text.color': TINTA,
    'xtick.color': TINTA_2,
    'ytick.color': TINTA_2,
    'axes.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 110,
    'savefig.dpi': DPI,
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

CLASES = ['unripe', 'ripe', 'overripe', 'damaged', 'empty']       # orden de presentación
NOMBRE_ES = {'unripe': 'Inmaduro', 'ripe': 'Maduro', 'overripe': 'Sobremaduro',
             'damaged': 'Dañado', 'empty': 'Vacío'}
EXT_IMG = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

VALORES = {}          # cifras que se copian al manuscrito
FIGURAS = []          # rutas de las figuras generadas


def guardar(fig, ruta):
    # Guarda la figura a 300 dpi y la registra.
    fig.savefig(ruta, dpi=DPI, bbox_inches='tight', pad_inches=0.04)
    FIGURAS.append(str(ruta))
    print('Guardada:', ruta.name)


def leer_rgb(ruta, lado_max=None):
    # Lee una imagen como arreglo RGB uint8, respetando la orientación EXIF.
    # Con lado_max, la reduce para que su lado mayor no supere ese valor.
    with Image.open(ruta) as im:
        if lado_max is not None and im.format == 'JPEG':
            im.draft('RGB', (lado_max * 2, lado_max * 2))     # decodificación rápida
        im = ImageOps.exif_transpose(im).convert('RGB')
        if lado_max is not None and max(im.size) > lado_max:
            im.thumbnail((lado_max, lado_max), Image.LANCZOS)
        return np.asarray(im)


def redimensionar(img, lado_mayor):
    # Redimensiona conservando la proporción hasta que el lado mayor mida lado_mayor.
    h, w = img.shape[:2]
    escala = lado_mayor / max(h, w)
    nuevo = (max(1, round(w * escala)), max(1, round(h * escala)))
    interp = cv2.INTER_AREA if escala < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, nuevo, interpolation=interp)


def recorte_cuadrado(img, lado=512):
    # Recorte cuadrado centrado, redimensionado a lado × lado píxeles.
    h, w = img.shape[:2]
    m = min(h, w)
    y0, x0 = (h - m) // 2, (w - m) // 2
    return cv2.resize(img[y0:y0 + m, x0:x0 + m], (lado, lado), interpolation=cv2.INTER_AREA)


def coma(x, dec=1):
    # Número con coma decimal, como se escribe en español.
    return f'{x:.{dec}f}'.replace('.', ',')


def eje_con_coma(eje, dec=2):
    eje.set_major_formatter(FuncFormatter(lambda v, _: coma(v, dec)))


def sin_ejes(ax):
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)


print('Bibliotecas listas. matplotlib', matplotlib.__version__, '| OpenCV', cv2.__version__)

from matplotlib.collections import LineCollection

ERRORES = {}
C_A, C_B = '#e8590c', '#ffd43b'
BANDA = 0.05                 # franja exterior del encuadre, como en el capítulo 8


def seccion(nombre, fn):
    # Ejecuta una sección y, si falla, anota el error y deja que las demás continúen.
    t0 = time.time()
    try:
        fn()
        print(f'[{nombre}] terminada en {time.time() - t0:.0f} s')
    except Exception:
        ERRORES[nombre] = traceback.format_exc()[-1500:]
        print(f'[{nombre}] ERROR\n{ERRORES[nombre]}')
    plt.close('all')


cap = cv2.VideoCapture(str(VIDEO))
FPS = float(cap.get(cv2.CAP_PROP_FPS)); NF = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'Video: {W} x {H} px, {FPS:.2f} fps, {NF} fotogramas, {NF / FPS:.1f} s')


def fotogramas(indices):
    # Lee el video una vez, en orden, y devuelve en RGB los fotogramas pedidos.
    pedir = set(int(i) for i in indices); sal = {}
    cap = cv2.VideoCapture(str(VIDEO)); k = 0
    while pedir:
        ok, f = cap.read()
        if not ok:
            break
        if k in pedir:
            sal[k] = cv2.cvtColor(f, cv2.COLOR_BGR2RGB); pedir.discard(k)
        k += 1
    cap.release()
    return sal


def gris(img, claro=0.25):
    # Versión en grises, algo aclarada, para que los trazos de color destaquen.
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = g * (1 - claro) + 255 * claro
    return np.dstack([g, g, g]).astype(np.uint8)


def preparar(df):
    # Añade a un registro de seguimiento el centro de cada caja y su tamaño (raíz del área).
    df = df.rename(columns={'confidence': 'conf', 'track_id': 'id'}).copy()
    df['cx'] = (df.x1 + df.x2) / 2; df['cy'] = (df.y1 + df.y2) / 2
    df['lado'] = np.sqrt((df.x2 - df.x1).clip(lower=1) * (df.y2 - df.y1).clip(lower=1))
    return df.sort_values(['id', 'frame']).reset_index(drop=True)


def iou_matriz(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return (inter / (aa[:, None] + ab[None, :] - inter + 1e-12)).astype(np.float32)


def en_borde(cx, cy):
    return (cx < BANDA * W) | (cx > (1 - BANDA) * W) | (cy < BANDA * H) | (cy > (1 - BANDA) * H)


CAJA = ['x1', 'y1', 'x2', 'y2']


def tabla_ids(df):
    # Una fila por identificador: cuándo aparece y desaparece, dónde, y cuánto se desplaza.
    g = df.groupby('id')
    pri = g.head(1).set_index('id'); ult = g.tail(1).set_index('id')
    t = pd.DataFrame({'f0': pri.frame, 'f1': ult.frame, 'n': g.size(), 'conf': g.conf.mean(), 'lado': g.lado.median(),
                      'cx0': pri.cx, 'cy0': pri.cy, 'cx1': ult.cx, 'cy1': ult.cy})
    for c in CAJA:
        t[c + '_0'] = pri[c]; t[c + '_1'] = ult[c]
    t['tramo'] = t.f1 - t.f0 + 1; t['tramo_s'] = t.tramo / FPS; t['cobertura'] = t.n / t.tramo
    t['neto'] = np.hypot(t.cx1 - t.cx0, t.cy1 - t.cy0)
    t['borde0'] = en_borde(t.cx0, t.cy0); t['borde1'] = en_borde(t.cx1, t.cy1)
    return t.reset_index()


def enlazar(t, hueco=30, iou_min=0.3):
    # Busca identificadores que continúan a otro: uno termina y, en menos de `hueco` fotogramas,
    # nace otro cuya primera caja se solapa con la última del anterior. Cada uno se usa una sola vez.
    fin = t[t.f1 < NF - 1].sort_values('f1'); nac = t[t.f0 > 0]
    cn = nac[[c + '_0' for c in CAJA]].to_numpy(np.float32); f0 = nac.f0.to_numpy(); idn = nac.id.to_numpy()
    libre = np.ones(len(nac), bool); pares = []
    for a in fin.itertuples():
        cand = np.flatnonzero(libre & (f0 > a.f1) & (f0 <= a.f1 + hueco))
        if len(cand) == 0:
            continue
        m = iou_matriz(np.array([[a.x1_1, a.y1_1, a.x2_1, a.y2_1]], np.float32), cn[cand])[0]
        j = int(m.argmax())
        if m[j] >= iou_min:
            libre[cand[j]] = False
            pares.append({'id_a': int(a.id), 'id_b': int(idn[cand[j]]), 'f1_a': int(a.f1), 'f0_b': int(f0[cand[j]]), 'iou': float(m[j])})
    return pd.DataFrame(pares, columns=['id_a', 'id_b', 'f1_a', 'f0_b', 'iou'])


def duplicados_al_nacer(df, t, iou_min=0.5):
    # Identificadores que nacen encima de otra ave que ya tiene identificador en ese mismo fotograma.
    por_frame = {int(k): g for k, g in df[df.frame.isin(t.f0[t.f0 > 0].unique())].groupby('frame')}
    dup = []
    for a in t[t.f0 > 0].itertuples():
        g = por_frame[int(a.f0)]; g = g[g.id != a.id]
        if len(g) and iou_matriz(np.array([[a.x1_0, a.y1_0, a.x2_0, a.y2_0]], np.float32), g[CAJA].to_numpy(np.float32)).max() >= iou_min:
            dup.append(int(a.id))
    return set(dup)


def clasificar(df):
    # Tabla de identificadores con el origen de cada uno.
    t = tabla_ids(df); enl = enlazar(t); dup = duplicados_al_nacer(df, t)
    cont = set(enl.id_b)
    def origen(r):
        if r.f0 == 0: return 'inicial'
        if r.id in cont: return 'continuacion'
        if r.id in dup: return 'duplicado'
        return 'borde' if r.borde0 else 'interior'
    t['origen'] = [origen(r) for r in t.itertuples()]
    return t, enl


def resumen_ids(df, t, enl):
    activos = df.groupby('frame').size().reindex(range(NF), fill_value=0)
    o = t.origen.value_counts().to_dict(); fin = t[t.f1 < NF - 1]
    return {'identificadores': int(len(t)), 'filas': int(len(df)), 'activos_media': float(activos.mean()), 'activos_min': int(activos.min()),
            'activos_max': int(activos.max()), 'completos': int((t.tramo == NF).sum()), 'filas_en_completos': float(t.n[t.tramo == NF].sum() / len(df)),
            'tramo_mediana_s': float(t.tramo_s.median()), 'menos_de_1s': int((t.tramo_s < 1).sum()), 'nacen_despues': int((t.f0 > 0).sum()),
            'terminan_antes': int(len(fin)), 'terminan_en_borde': int(fin.borde1.sum()), 'origen': {k: int(v) for k, v in o.items()},
            'enlaces': int(len(enl)), 'hueco_mediano_enlace': float((enl.f0_b - enl.f1_a).median()) if len(enl) else None,
            'aves_estimadas': int(len(t) - len(enl) - o.get('duplicado', 0)), 'ids_por_ave_visible': float(len(t) / max(activos.mean(), 1e-9)),
            'conf_completos': float(t.conf[t.tramo == NF].mean()) if (t.tramo == NF).any() else None,
            'conf_menos_de_1s': float(t.conf[t.tramo_s < 1].mean()) if (t.tramo_s < 1).any() else None}


TR = preparar(pd.read_csv(TRACKS_CSV))
TI, ENL = clasificar(TR)
VALORES['video'] = {'ancho': W, 'alto': H, 'fps': FPS, 'fotogramas': NF}
VALORES['proyecto'] = resumen_ids(TR, TI, ENL)
print(VALORES['proyecto'])
TI.round(2).to_csv(DIR_VAL / 'cap09_identificadores.csv', index=False); ENL.round(3).to_csv(DIR_VAL / 'cap09_enlaces.csv', index=False)


def guardar_npz(df, ruta):
    np.savez_compressed(ruta, frame=df.frame.to_numpy(np.int16), id=df.id.to_numpy(np.int32),
                        caja=np.round(df[CAJA].to_numpy(np.float32) * 10).astype(np.int16), conf=np.round(df.conf.to_numpy() * 255).astype(np.uint8))


guardar_npz(TR, DIR_VAL / 'cap09_registro_proyecto.npz')

## 1. El video, fotograma a fotograma

In [ ]:
def s_video():
    # Tipo de cada fotograma (I, P o B) según el códec, diferencia con el anterior y nitidez.
    tipos = []
    try:
        sal = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries', 'frame=pict_type', '-of', 'csv=p=0', str(VIDEO)],
                             capture_output=True, text=True, timeout=300).stdout
        tipos = [l.strip().strip(',') for l in sal.splitlines() if l.strip()]
    except Exception as e:
        print('ffprobe no disponible:', e)
    cap = cv2.VideoCapture(str(VIDEO)); ant = None; dif, nit = [], []; guardados = {}
    quiero = sorted(set([0, NF // 4, NF // 2, 3 * NF // 4, NF - 1]))
    k = 0
    while True:
        ok, f = cap.read()
        if not ok:
            break
        if k in quiero:
            cv2.imwrite(str(DIR_EXP / f'fotograma_{k:03d}.jpg'), f, [cv2.IMWRITE_JPEG_QUALITY, 90])
        g = cv2.cvtColor(cv2.resize(f, (W // 2, H // 2), interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2GRAY).astype(np.float32)
        dif.append(float(np.abs(g - ant).mean()) if ant is not None else np.nan); nit.append(float(cv2.Laplacian(g, cv2.CV_32F).var()))
        ant = g; k += 1
    cap.release()
    # Desplazamiento de las cajas entre fotogramas consecutivos: mediana del módulo y deriva común (dx, dy).
    a = TR[['id', 'frame', 'cx', 'cy']]; b = a.copy(); b['frame'] += 1
    m = a.merge(b, on=['id', 'frame'], suffixes=('', '_ant'))
    m['dx'] = m.cx - m.cx_ant; m['dy'] = m.cy - m.cy_ant; m['d'] = np.hypot(m.dx, m.dy)
    g = m.groupby('frame').agg(d_mediana=('d', 'median'), dx_mediana=('dx', 'median'), dy_mediana=('dy', 'median')).reindex(range(k))
    t = pd.DataFrame({'frame': range(k), 'tipo': (tipos + [''] * k)[:k], 'diferencia': dif, 'nitidez': nit,
                      'aves': TR.groupby('frame').size().reindex(range(k), fill_value=0).values})
    t = pd.concat([t, g.reset_index(drop=True)], axis=1)
    t.round(4).to_csv(DIR_VAL / 'cap09_fotogramas.csv', index=False)
    picos = t.frame[t.d_mediana > 2.5 * t.d_mediana.median()].tolist()
    clave = t.frame[t.tipo == 'I'].tolist()
    VALORES['video'].update({'fotogramas_leidos': k, 'tipos': {x: int((t.tipo == x).sum()) for x in sorted(set(t.tipo))}, 'fotogramas_clave': clave[:60],
                             'picos_desplazamiento': picos[:60], 'd_mediana_px': float(t.d_mediana.median()),
                             'd_en_picos_px': float(t.d_mediana[t.frame.isin(picos)].mean()) if picos else None,
                             'diferencia_media': float(np.nanmean(dif)), 'diferencia_en_picos': float(t.diferencia[t.frame.isin(picos)].mean()) if picos else None,
                             'fotogramas_repetidos': int((t.diferencia < 0.05).sum())})
    print({k2: v for k2, v in VALORES['video'].items()})

seccion('video', s_video)

## 2. Lo que entrega un algoritmo de seguimiento

Un mismo recorte del encuadre en tres instantes, con el identificador que el algoritmo asignó a cada ave.

In [ ]:
def s_identidades():
    inst = [0, int(round(FPS)), int(round(5 * FPS))]; inst = [min(i, NF - 1) for i in inst]
    fr = fotogramas(inst)
    x0, y0, aw, ah = W // 2 - W // 6, H // 2 - H // 6, W // 3, H // 3
    fig, ejes = plt.subplots(1, 3, figsize=(ANCHO_IN, ANCHO_IN / 3 * ah / aw + 0.25))
    ids0 = set(TR.id[(TR.frame == inst[0])])
    for ax, f in zip(ejes, inst):
        ax.imshow(fr[f][y0:y0 + ah, x0:x0 + aw]); sin_ejes(ax)
        g = TR[(TR.frame == f) & (TR.cx > x0) & (TR.cx < x0 + aw) & (TR.cy > y0) & (TR.cy < y0 + ah)]
        for r in g.itertuples():
            nuevo = r.id not in ids0
            ax.add_patch(Rectangle((r.x1 - x0, r.y1 - y0), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=0.6, ec=C_A if nuevo else 'white'))
            ax.text(r.cx - x0, r.cy - y0, str(int(r.id)), ha='center', va='center', fontsize=4.5, color='white',
                    bbox=dict(boxstyle='round,pad=0.12', fc=C_A if nuevo else TINTA, ec='none', alpha=0.8))
        ax.set_xlim(0, aw); ax.set_ylim(ah, 0)
        ax.set_title(f't = {coma(f / FPS, 1)} s', pad=3)
    fig.subplots_adjust(wspace=0.03, left=0.005, right=0.995, top=0.9, bottom=0.01)
    guardar(fig, DIR_FIG / 'Fig_9_1_identidades.png'); plt.show()
    en = lambda f: set(TR.id[(TR.frame == f) & (TR.cx > x0) & (TR.cx < x0 + aw) & (TR.cy > y0) & (TR.cy < y0 + ah)])
    a, b, c = en(inst[0]), en(inst[1]), en(inst[2])
    VALORES['identidades'] = {'instantes': inst, 'recorte': [x0, y0, aw, ah], 'aves': [len(a), len(b), len(c)],
                              'conservan_1s': len(a & b), 'conservan_5s': len(a & c), 'nuevos_5s': len(c - ids0)}
    print(VALORES['identidades'])

seccion('identidades', s_identidades)

## 3. Dónde nacen y dónde terminan los identificadores

Cada identificador que aparece después del primer fotograma se clasifica según su origen: continúa a otro que se acababa de perder, duplica a un ave que ya tenía identificador, aparece en la franja exterior del encuadre o aparece en el interior.

In [ ]:
def s_fragmentacion():
    fr = fotogramas([NF // 2]); base = gris(fr[NF // 2], 0.35)
    fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_IN, ANCHO_IN / 2 * H / W + 0.3))
    nac = TI[TI.f0 > 0]; fin = TI[TI.f1 < NF - 1]
    for ax, t, cx, cy, tit in ((ejes[0], nac, 'cx0', 'cy0', f'(a) Dónde aparecen ({len(nac)})'), (ejes[1], fin, 'cx1', 'cy1', f'(b) Dónde terminan ({len(fin)})')):
        ax.imshow(base); sin_ejes(ax)
        ax.add_patch(Rectangle((BANDA * W, BANDA * H), (1 - 2 * BANDA) * W, (1 - 2 * BANDA) * H, fill=False, lw=0.6, ls=(0, (4, 3)), ec=TINTA))
        ax.scatter(t[cx], t[cy], s=7, c=C_A, edgecolors=TINTA, linewidths=0.25)
        ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_title(tit, pad=3)
    fig.subplots_adjust(wspace=0.03, left=0.005, right=0.995, top=0.9, bottom=0.01)
    guardar(fig, DIR_FIG / 'Fig_9_3_nacen_y_terminan.png'); plt.show()
    # Tramo y confianza según el origen.
    por = TI.groupby('origen').agg(n=('id', 'size'), tramo_mediana_s=('tramo_s', 'median'), conf=('conf', 'mean'), lado=('lado', 'median'),
                                   en_borde=('borde0', 'mean')).round(3)
    print(por)
    VALORES['fragmentacion'] = {'por_origen': por.reset_index().to_dict('records'),
                                'area_franja': float(1 - (1 - 2 * BANDA) ** 2),
                                'nacen_en_franja': float(nac.borde0.mean()), 'terminan_en_franja': float(fin.borde1.mean()),
                                'filas_en_franja': float(en_borde(TR.cx, TR.cy).mean()),
                                'nacen_por_segundo': float(len(nac) / (NF / FPS)),
                                'cobertura_menor_090': int((TI.cobertura < 0.9).sum())}
    # Solo la zona interior: cuántos identificadores hacen falta para las aves que hay.
    d = TR[~en_borde(TR.cx, TR.cy)]
    VALORES['fragmentacion']['interior'] = {'identificadores': int(d.id.nunique()), 'activos_media': float(d.groupby('frame').size().mean()),
                                            'completos': int((d.groupby('id').size() == NF).sum())}
    print(VALORES['fragmentacion']['interior'])

seccion('fragmentacion', s_fragmentacion)

## 4. Seguir a un ave durante veinte segundos

Tres aves con identificador completo, vistas cada cinco segundos, y un caso en que el identificador cambia.

In [ ]:
def s_seguir():
    comp = TI[TI.tramo == NF].sort_values('neto')
    assert len(comp) >= 3, 'No hay tres identificadores completos.'
    elegidos = [int(comp.id.iloc[-1]), int(comp.id.iloc[len(comp) // 2]), int(comp.id.iloc[0])]
    inst = [min(int(round(s * FPS)), NF - 1) for s in (0, 5, 10, 15, 20)]
    # Cambio de identificador: enlace en el interior, entre dos identificadores largos.
    ta = TI[['id', 'tramo', 'borde1', 'cx1', 'cy1']].rename(columns={'id': 'id_a', 'tramo': 'tramo_a'})
    tb = TI[['id', 'tramo']].rename(columns={'id': 'id_b', 'tramo': 'tramo_b'})
    cand = ENL.merge(ta, on='id_a').merge(tb, on='id_b')
    cand = cand[(~cand.borde1) & (cand.tramo_a >= FPS) & (cand.tramo_b >= FPS)].sort_values('iou', ascending=False)
    cambio = cand.iloc[0] if len(cand) else None
    inst_c = []
    if cambio is not None:
        inst_c = sorted(set(int(np.clip(v, 0, NF - 1)) for v in (cambio.f1_a - int(FPS), cambio.f1_a, cambio.f0_b, cambio.f0_b + int(FPS), cambio.f0_b + int(3 * FPS))))
    fr = fotogramas(inst + inst_c)
    lado = 220; filas = 3 + (1 if cambio is not None else 0)
    fig, ejes = plt.subplots(filas, 5, figsize=(ANCHO_IN, ANCHO_IN / 5 * filas + 0.3))

    def recorte(img, cx, cy, lado=lado):
        x0 = int(np.clip(cx - lado / 2, 0, W - lado)); y0 = int(np.clip(cy - lado / 2, 0, H - lado))
        return img[y0:y0 + lado, x0:x0 + lado], x0, y0

    for i, tid in enumerate(elegidos):
        g = TR[TR.id == tid].set_index('frame')
        # El recorte queda fijo y abarca todo el recorrido del ave.
        ext = max(g.cx.max() - g.cx.min(), g.cy.max() - g.cy.min()); l_i = int(min(max(lado, ext + 140), H))
        mx, my = (g.cx.max() + g.cx.min()) / 2, (g.cy.max() + g.cy.min()) / 2
        for j, f in enumerate(inst):
            ax = ejes[i, j]; sin_ejes(ax)
            img, x0, y0 = recorte(fr[f], mx, my, l_i); ax.imshow(img)
            if f in g.index:
                r = g.loc[f]; ax.add_patch(Rectangle((r.x1 - x0, r.y1 - y0), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=1.0, ec=C_B))
            ax.set_xlim(0, l_i); ax.set_ylim(l_i, 0)
            if i == 0:
                ax.set_title(f't = {f / FPS:.0f} s', pad=3)
            if j == 0:
                ax.set_ylabel(f'Ave {tid}', fontsize=7)
    if cambio is not None:
        ga = TR[TR.id == cambio.id_a].set_index('frame'); gb = TR[TR.id == cambio.id_b].set_index('frame')
        for j in range(5):
            ax = ejes[3, j]; sin_ejes(ax)
            if j >= len(inst_c):
                ax.axis('off'); continue
            f = inst_c[j]; img, x0, y0 = recorte(fr[f], cambio.cx1, cambio.cy1); ax.imshow(img)
            for g, col, tid in ((ga, C_B, cambio.id_a), (gb, C_A, cambio.id_b)):
                if f in g.index:
                    r = g.loc[f]; ax.add_patch(Rectangle((r.x1 - x0, r.y1 - y0), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=1.0, ec=col))
                    ax.text(r.x1 - x0 + 3, r.y1 - y0 + 4, str(int(tid)), ha='left', va='top', fontsize=6, color='white', bbox=dict(boxstyle='round,pad=0.15', fc=TINTA, ec='none', alpha=0.8))
            ax.set_xlim(0, lado); ax.set_ylim(lado, 0); ax.set_xlabel(f't = {coma(f / FPS, 2)} s', fontsize=6.5, labelpad=2)
            if j == 0:
                ax.set_ylabel('Cambio', fontsize=7)
    fig.subplots_adjust(wspace=0.03, hspace=0.05, left=0.03, right=0.995, top=0.95, bottom=0.03)
    guardar(fig, DIR_FIG / 'Fig_9_4_seguir_un_ave.png'); plt.show()
    VALORES['seguir'] = {'aves': elegidos, 'neto_px': [float(TI.neto[TI.id == t].iloc[0]) for t in elegidos], 'lado_px': [float(TI.lado[TI.id == t].iloc[0]) for t in elegidos],
                         'instantes': inst, 'cambio': None if cambio is None else {'id_a': int(cambio.id_a), 'id_b': int(cambio.id_b), 'f1_a': int(cambio.f1_a), 'f0_b': int(cambio.f0_b), 'iou': float(cambio.iou), 'instantes': inst_c},
                         'cambios_interiores_largos': int(len(cand))}
    print(VALORES['seguir'])

seccion('seguir', s_seguir)

## 5. Trayectorias

In [ ]:
def s_trayectorias():
    fr = fotogramas([NF - 1]); base = gris(fr[NF - 1], 0.3)
    largos = TI[TI.n >= FPS]
    lineas, colores = [], []
    for tid, g in TR[TR.id.isin(largos.id)].groupby('id'):
        lineas.append(g[['cx', 'cy']].to_numpy()); colores.append(ACENTO if len(g) == NF else C_A)
    # Recorte con más desplazamiento neto acumulado.
    gx, gy = 3, 3; aw, ah = W // gx, H // gy
    suma = np.zeros((gy, gx))
    for r in largos.itertuples():
        suma[min(int(r.cy1 // ah), gy - 1), min(int(r.cx1 // aw), gx - 1)] += r.neto
    jy, jx = np.unravel_index(suma.argmax(), suma.shape); x0, y0 = jx * aw, jy * ah
    fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_IN, ANCHO_IN / 2 * H / W + 0.3))
    for ax, lw in ((ejes[0], 0.5), (ejes[1], 1.0)):
        ax.imshow(base); sin_ejes(ax)
        ax.add_collection(LineCollection(lineas, colors=colores, linewidths=lw))
    ejes[0].add_patch(Rectangle((x0, y0), aw, ah, fill=False, lw=0.8, ec=TINTA))
    ejes[0].set_xlim(0, W); ejes[0].set_ylim(H, 0); ejes[0].set_title('(a) Encuadre completo', pad=3)
    ejes[1].set_xlim(x0, x0 + aw); ejes[1].set_ylim(y0 + ah, y0); ejes[1].set_title('(b) Ampliación del recuadro', pad=3)
    fig.subplots_adjust(wspace=0.03, left=0.005, right=0.995, top=0.9, bottom=0.01)
    guardar(fig, DIR_FIG / 'Fig_9_5_trayectorias.png'); plt.show()
    comp = TI[TI.tramo == NF]; rel = comp.neto / comp.lado
    VALORES['trayectorias'] = {'dibujadas': int(len(lineas)), 'recorte': [int(x0), int(y0), int(aw), int(ah)],
                               'neto_px': {q: float(comp.neto.quantile(v)) for q, v in (('p10', .1), ('p25', .25), ('p50', .5), ('p75', .75), ('p90', .9), ('max', 1))},
                               'neto_cuerpos': {q: float(rel.quantile(v)) for q, v in (('p10', .1), ('p25', .25), ('p50', .5), ('p75', .75), ('p90', .9), ('max', 1))},
                               'mas_de_medio_cuerpo': float((rel > 0.5).mean()), 'mas_de_un_cuerpo': float((rel > 1).mean()), 'mas_de_dos_cuerpos': float((rel > 2).mean()),
                               'lado_mediano_px': float(TR.lado.median())}
    print(VALORES['trayectorias'])

seccion('trayectorias', s_trayectorias)

## 6. Medir el movimiento

El desplazamiento del centro de cada caja se calcula con intervalos de 1 fotograma a 10 segundos. Si lo que se mide es el temblor de las cajas, el desplazamiento no crece con el intervalo; si es movimiento real, sí.

In [ ]:
def pares(L):
    a = TR[['id', 'frame', 'cx', 'cy', 'lado']]; b = a[['id', 'frame', 'cx', 'cy']].copy(); b['frame'] += L
    m = a.merge(b, on=['id', 'frame'], suffixes=('', '_ant'))
    m['d'] = np.hypot(m.cx - m.cx_ant, m.cy - m.cy_ant); m['rel'] = m.d / m.lado
    return m


def s_movimiento():
    filas = []
    for L in (1, 2, 3, 5, 10, 15, 30, 60, 150, 300):
        if L >= NF:
            continue
        m = pares(L)
        if len(m) == 0:
            continue
        filas.append({'fotogramas': L, 'segundos': L / FPS, 'pares': int(len(m)), 'd_p25': m.d.quantile(.25), 'd_mediana': m.d.median(), 'd_p75': m.d.quantile(.75),
                      'd_p90': m.d.quantile(.9), 'd_p99': m.d.quantile(.99), 'v_mediana_px_s': m.d.median() * FPS / L, 'v_p90_px_s': m.d.quantile(.9) * FPS / L,
                      'rel_mediana': m.rel.median(), 'rel_p90': m.rel.quantile(.9), 'mas_de_medio_cuerpo': float((m.rel > 0.5).mean()), 'mas_de_un_cuerpo': float((m.rel > 1).mean())})
    t = pd.DataFrame(filas); t.round(4).to_csv(DIR_VAL / 'cap09_movimiento_intervalos.csv', index=False); print(t.round(3).to_string(index=False))
    # Temblor: en las aves completas, dispersión del centro alrededor de su media móvil de 1 segundo.
    comp = TR[TR.id.isin(TI.id[TI.tramo == NF])]
    v = int(FPS) | 1
    res = comp.groupby('id')[['cx', 'cy']].transform(lambda s: s - s.rolling(v, center=True, min_periods=v // 2).mean())
    temblor = np.sqrt((res ** 2).sum(axis=1)).groupby(comp.id).median()
    # Serie por fotograma con intervalos de 1 fotograma, 0,5 s y 2 s.
    serie = pd.DataFrame({'frame': range(NF)})
    L2 = int(round(2 * FPS))
    for nombre, L in (('lag1', 1), ('lag05s', int(round(FPS / 2))), ('lag2s', L2)):
        m = pares(L); g = m.groupby('frame')
        serie[f'v_mediana_{nombre}'] = (g.d.median() * FPS / L).reindex(range(NF)).values
        if nombre == 'lag2s':
            serie['en_movimiento_lag2s'] = g.rel.apply(lambda s: float((s > 0.5).mean())).reindex(range(NF)).values
    serie.round(4).to_csv(DIR_VAL / 'cap09_movimiento_por_fotograma.csv', index=False)
    # Tamaño de las aves según la altura en la imagen (perspectiva).
    bandas = pd.cut(TR.cy, np.linspace(0, H, 7), labels=False)
    tam = TR.groupby(bandas).lado.median()
    m2 = pares(L2)
    VALORES['movimiento'] = {'intervalos': t.to_dict('records'), 'temblor_px_mediana': float(temblor.median()), 'temblor_px_p90': float(temblor.quantile(.9)),
                             'temblor_relativo': float((temblor / TI.set_index('id').lado.reindex(temblor.index)).median()),
                             'lado_por_banda_vertical': [float(x) for x in tam.reindex(range(6)).values],
                             'en_movimiento_2s': {'media': float(serie.en_movimiento_lag2s.mean()), 'min': float(serie.en_movimiento_lag2s.min()), 'max': float(serie.en_movimiento_lag2s.max())},
                             'umbrales_2s': {str(u): float((m2.rel > u).mean()) for u in (0.1, 0.25, 0.5, 1.0, 2.0)}}
    print({k: v for k, v in VALORES['movimiento'].items() if k != 'intervalos'})
    # Figura: aves que se desplazan más de medio cuerpo en dos segundos.
    f = min(int(round(10 * FPS)), NF - 1); fr = fotogramas([f]); base = gris(fr[f], 0.3)
    act = m2[m2.frame == f]; mov = set(act.id[act.rel > 0.5]); g = TR[TR.frame == f]
    fig, ax = plt.subplots(figsize=(ANCHO_IN, ANCHO_IN * H / W)); ax.imshow(base); sin_ejes(ax)
    for r in g.itertuples():
        es = r.id in mov
        ax.add_patch(Rectangle((r.x1, r.y1), r.x2 - r.x1, r.y2 - r.y1, fill=False, lw=1.1 if es else 0.5, ec=C_A if es else ACENTO))
    estelas = [h[['cx', 'cy']].to_numpy() for tid, h in TR[TR.id.isin(mov) & (TR.frame > f - L2) & (TR.frame <= f)].groupby('id')]
    ax.add_collection(LineCollection(estelas, colors=C_B, linewidths=1.2))
    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    fig.subplots_adjust(left=0.005, right=0.995, top=0.995, bottom=0.005)
    guardar(fig, DIR_FIG / 'Fig_9_7_aves_en_movimiento.png'); plt.show()
    VALORES['movimiento']['figura'] = {'fotograma': f, 'aves': int(len(g)), 'con_historia': int(len(act)), 'en_movimiento': int(len(mov))}
    print(VALORES['movimiento']['figura'])

seccion('movimiento', s_movimiento)

## 7. Zonas y mapa de concentración

In [ ]:
def s_zonas():
    f = NF // 2; fr = fotogramas([f])
    zx = (TR.cx >= W / 2).astype(int); zy = (TR.cy >= H / 2).astype(int); zona = (zy * 2 + zx + 1)
    cuenta = pd.crosstab(TR.frame, zona).reindex(index=range(NF), columns=[1, 2, 3, 4], fill_value=0)
    cuenta.columns = ['Z1', 'Z2', 'Z3', 'Z4']; cuenta.to_csv(DIR_VAL / 'cap09_zonas_por_fotograma.csv')
    # Fracción del suelo de cada zona cubierta por cajas, y tamaño de las aves en cada zona.
    esc = 4; cub = []
    for k in range(0, NF, 10):
        mk = np.zeros((H // esc, W // esc), np.uint8)
        for r in TR[TR.frame == k][CAJA].to_numpy():
            cv2.rectangle(mk, (int(r[0] / esc), int(r[1] / esc)), (int(r[2] / esc), int(r[3] / esc)), 1, -1)
        h2, w2 = mk.shape[0] // 2, mk.shape[1] // 2
        cub.append([mk[:h2, :w2].mean(), mk[:h2, w2:].mean(), mk[h2:, :w2].mean(), mk[h2:, w2:].mean(), mk.mean()])
    cub = np.array(cub).mean(0)
    lado_z = TR.groupby(zona).lado.median().reindex([1, 2, 3, 4])
    # Rejilla fina: aves por celda y por fotograma.
    nx, ny = 16, 9
    ix = np.clip((TR.cx / W * nx).astype(int), 0, nx - 1); iy = np.clip((TR.cy / H * ny).astype(int), 0, ny - 1)
    def rej(m):
        g = np.zeros((ny, nx)); np.add.at(g, (iy[m].to_numpy(), ix[m].to_numpy()), 1); return g
    total = rej(np.ones(len(TR), bool)) / NF
    mitad1 = rej((TR.frame < NF // 2).to_numpy()); mitad2 = rej((TR.frame >= NF // 2).to_numpy())
    uno = [np.corrcoef(rej((TR.frame == k).to_numpy()).ravel(), total.ravel())[0, 1] for k in range(0, NF, 30)]
    interior = np.zeros((ny, nx), bool); interior[1:-1, 1:-1] = True
    VALORES['zonas'] = {'media': {z: float(cuenta[z].mean()) for z in cuenta}, 'desviacion': {z: float(cuenta[z].std()) for z in cuenta},
                        'porcentaje': {z: float(cuenta[z].sum() / cuenta.values.sum() * 100) for z in cuenta},
                        'cubierta': dict(zip(['Z1', 'Z2', 'Z3', 'Z4', 'total'], [float(x) for x in cub])),
                        'lado_mediano': {f'Z{k}': float(v) for k, v in lado_z.items()},
                        'rejilla': total.round(3).tolist(), 'correlacion_mitades': float(np.corrcoef(mitad1.ravel(), mitad2.ravel())[0, 1]),
                        'correlacion_un_fotograma': float(np.mean(uno)), 'celda_max': float(total.max()), 'celda_min': float(total.min()),
                        'celda_interior_cv': float(total[interior].std() / total[interior].mean()),
                        'celdas_borde_media': float(total[~interior].mean()), 'celdas_interior_media': float(total[interior].mean())}
    print({k: v for k, v in VALORES['zonas'].items() if k != 'rejilla'})
    # Mapa de concentración: centros acumulados y suavizados.
    hw, hh = W // 6, H // 6
    mapa = np.zeros((hh, hw), np.float32)
    np.add.at(mapa, (np.clip((TR.cy / H * hh).astype(int), 0, hh - 1).to_numpy(), np.clip((TR.cx / W * hw).astype(int), 0, hw - 1).to_numpy()), 1)
    mapa = cv2.GaussianBlur(mapa, (0, 0), sigmaX=7, sigmaY=7); mapa /= max(float(mapa.max()), 1e-9)
    np.save(DIR_EXP / 'mapa_concentracion.npy', mapa.astype(np.float16))
    fig, ejes = plt.subplots(1, 2, figsize=(ANCHO_IN, ANCHO_IN / 2 * H / W + 0.3))
    ax = ejes[0]; ax.imshow(fr[f]); sin_ejes(ax)
    ax.plot([W / 2, W / 2], [0, H], color='white', lw=0.9); ax.plot([0, W], [H / 2, H / 2], color='white', lw=0.9)
    for z, (px, py) in zip(['Z1', 'Z2', 'Z3', 'Z4'], [(.25, .25), (.75, .25), (.25, .75), (.75, .75)]):
        ax.text(px * W, py * H, f"{z}\n{coma(VALORES['zonas']['media'][z], 0)} aves", ha='center', va='center', fontsize=7, color='white',
                bbox=dict(boxstyle='round,pad=0.3', fc=TINTA, ec='none', alpha=0.75))
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_title('(a) Cuatro zonas', pad=3)
    ax = ejes[1]; ax.imshow(gris(fr[f], 0.2)); sin_ejes(ax)
    im = ax.imshow(mapa, cmap='inferno', alpha=0.6, extent=[0, W, H, 0], vmin=0, vmax=1)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_title('(b) Concentración acumulada en 20 s', pad=3)
    fig.subplots_adjust(wspace=0.03, left=0.005, right=0.995, top=0.9, bottom=0.01)
    guardar(fig, DIR_FIG / 'Fig_9_8_zonas_y_mapa.png'); plt.show()

seccion('zonas', s_zonas)

## 8. Experimento: dos algoritmos y varias configuraciones

Se repite el seguimiento de los 20 segundos con el mismo detector y los mismos parámetros del proyecto (960 píxeles, confianza 0,25, IoU de 0,60), y se cambia en cada pasada una sola cosa. Como no hay identidades anotadas a mano, se comparan indicadores de funcionamiento: cuántos identificadores hacen falta, cuántos cubren todo el video, cuántos son continuación de otro.

In [ ]:
import yaml
DIR_CFG = Path(ultralytics.__file__).parent / 'cfg' / 'trackers'
VARIANTES = [
    ('bytetrack', 'bytetrack.yaml', {}, 0.25),
    ('botsort', 'botsort.yaml', {}, 0.25),
    ('botsort_sin_gmc', 'botsort.yaml', {'gmc_method': 'none'}, 0.25),
    ('bytetrack_memoria90', 'bytetrack.yaml', {'track_buffer': 90}, 0.25),
    ('botsort_memoria90', 'botsort.yaml', {'track_buffer': 90}, 0.25),
    ('bytetrack_conf010', 'bytetrack.yaml', {}, 0.10),
    ('botsort_nuevo060', 'botsort.yaml', {'new_track_thresh': 0.60}, 0.25),
    ('botsort_reid', 'botsort.yaml', {'with_reid': True}, 0.25),
    ('ocsort', 'ocsort.yaml', {}, 0.25),
]


def seguir_video(nombre, base, cambios, conf):
    ruta = DIR_EXP / f'{nombre}.npz'; rj = DIR_EXP / f'{nombre}.json'
    if ruta.exists() and rj.exists():
        print('REUTILIZADO:', nombre)
    else:
        cfg = yaml.safe_load((DIR_CFG / base).read_text(encoding='utf-8'))
        for k in cambios:
            assert k in cfg, f'El archivo {base} no tiene el parámetro {k}'
        cfg.update(cambios)
        ryaml = LOCAL / f'cap09_{nombre}.yaml'; ryaml.write_text(yaml.safe_dump(cfg), encoding='utf-8')
        modelo = YOLO(str(MOD_PIO)); cap = cv2.VideoCapture(str(VIDEO)); filas = []; k = 0; t0 = time.time(); t1 = None
        while True:
            ok, f = cap.read()
            if not ok:
                break
            r = modelo.track(source=f, persist=True, tracker=str(ryaml), imgsz=960, conf=conf, iou=0.60, max_det=1000, device=DISP, verbose=False)[0]
            if r.boxes is not None and r.boxes.id is not None and len(r.boxes) > 0:
                c = r.boxes.xyxy.cpu().numpy(); ids = r.boxes.id.int().cpu().numpy(); cf = r.boxes.conf.cpu().numpy()
                filas.append(np.column_stack([np.full(len(ids), k), ids, c, cf]))
            k += 1
            if k == 10:
                t1 = time.time()                      # los primeros fotogramas incluyen la carga del modelo
        cap.release()
        fps = (k - 10) / (time.time() - t1) if t1 else None
        a = np.concatenate(filas); df = pd.DataFrame(a, columns=['frame', 'id'] + CAJA + ['conf'])
        df['frame'] = df.frame.astype(int); df['id'] = df.id.astype(int)
        guardar_npz(df, ruta)
        rj.write_text(json.dumps({'fps': fps, 'segundos': time.time() - t0, 'configuracion': cfg, 'conf': conf, 'gpu': GPU}, default=str), encoding='utf-8')
        del modelo; gc.collect()
        if HAY_GPU:
            torch.cuda.empty_cache()
    z = np.load(ruta); info = json.loads(rj.read_text(encoding='utf-8'))
    df = pd.DataFrame(z['caja'].astype(np.float32) / 10, columns=CAJA); df['frame'] = z['frame']; df['id'] = z['id']; df['conf'] = z['conf'] / 255
    df = preparar(df); t, enl = clasificar(df); r = resumen_ids(df, t, enl)
    r.update({'fps': info.get('fps'), 'conf': conf, 'cambios': cambios, 'base': base})
    return r


def s_seguimiento():
    if not REPETIR_SEGUIMIENTO:
        print('Experimento omitido.'); return
    VALORES['seguimiento'] = {}; VALORES['seguimiento_errores'] = {}
    # Detecciones por fotograma sin seguimiento, como referencia del conteo.
    rd = DIR_EXP / 'detecciones_por_fotograma.csv'
    if not rd.exists():
        modelo = YOLO(str(MOD_PIO)); cap = cv2.VideoCapture(str(VIDEO)); n25, n10 = [], []
        while True:
            ok, f = cap.read()
            if not ok:
                break
            cf = modelo.predict(source=f, imgsz=960, conf=0.10, iou=0.60, max_det=1000, device=DISP, verbose=False)[0].boxes.conf.cpu().numpy()
            n25.append(int((cf >= 0.25).sum())); n10.append(int(len(cf)))
        cap.release(); pd.DataFrame({'frame': range(len(n25)), 'conf_025': n25, 'conf_010': n10}).to_csv(rd, index=False)
        del modelo; gc.collect()
    d = pd.read_csv(rd); VALORES['detecciones_sin_seguimiento'] = {'conf_025_media': float(d.conf_025.mean()), 'conf_010_media': float(d.conf_010.mean())}
    for nombre, base, cambios, conf in VARIANTES:
        try:
            r = seguir_video(nombre, base, cambios, conf); VALORES['seguimiento'][nombre] = r
            print(f"{nombre:22s} ids {r['identificadores']:4d} | activos {r['activos_media']:6.1f} | completos {r['completos']:4d} | enlaces {r['enlaces']:4d} | "
                  f"mediana {r['tramo_mediana_s']:5.1f} s | fps {r['fps'] if r['fps'] is None else round(r['fps'], 1)}")
        except Exception:
            VALORES['seguimiento_errores'][nombre] = traceback.format_exc()[-1200:]
            print(f'{nombre}: ERROR\n', VALORES['seguimiento_errores'][nombre])

seccion('seguimiento', s_seguimiento)

## 9. Guardar y descargar

In [ ]:
VALORES['errores'] = ERRORES
VALORES['entorno'] = {'gpu': GPU, 'torch': torch.__version__, 'ultralytics': ultralytics.__version__}
ruta_json = DIR_VAL / 'valores_cap09.json'
ruta_json.write_text(json.dumps(VALORES, ensure_ascii=False, indent=1, default=str), encoding='utf-8')
ruta_zip = LOCAL / 'AgroVision_cap09.zip'
with zipfile.ZipFile(ruta_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(DIR_FIG.glob('Fig_9_*.png')):
        z.write(f, 'cap09/' + f.name)
    for f in [ruta_json] + sorted(DIR_VAL.glob('cap09_*.csv')) + sorted(DIR_VAL.glob('cap09_*.npz')):
        z.write(f, f.name)
    for f in sorted(DIR_EXP.iterdir()):
        if f.is_file():
            z.write(f, 'cap09_seguimiento/' + f.name)
print('Secciones con error:', list(ERRORES) or 'ninguna')
print('Figuras:', len(FIGURAS), '· zip de', round(ruta_zip.stat().st_size / 1e6, 1), 'MB')
if EN_COLAB:
    from google.colab import files
    files.download(str(ruta_zip))